### STEP 0. 제출용 Notebook과 Chapter10 입력 준비

In [1]:
from pathlib import Path
import pandas as pd

# 노트북 실행 위치가 프로젝트 루트인지 notebooks 폴더인지 확인한다.
project_root = Path.cwd()
if not (project_root / "data" / "processed" / "orders_clean.csv").exists():
    project_root = project_root.parent

# 전처리된 주문 데이터를 읽는다.
orders = pd.read_csv(project_root / "data" / "processed" / "orders_clean.csv")

# 데이터 크기, 컬럼, 주문 상태별 건수를 확인한다.
print("주문 데이터 크기:", orders.shape)
print("컬럼:", orders.columns.tolist())
print("\n주문 상태별 건수:")
print(orders["order_status"].value_counts(dropna=False))

주문 데이터 크기: (300, 7)
컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

주문 상태별 건수:
order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64


### STEP 1. Target Contract와 예측 시점 정의

### STEP 1. 타깃과 예측 시점

- **예측 대상:** 주문이 취소될지 여부
- **예측 단위:** 주문 1건
- **타깃 정의:** `completed`는 `is_cancelled = 0`, `cancelled`는 `is_cancelled = 1`로 둔다. `refunded`와 기타 상태는 이번 모델링에서 제외한다.
- **예측 시점:** 주문이 생성된 직후
- **사용 가능한 정보:** 주문 생성 직후 이미 확인할 수 있는 주문상품의 수량·가격과 그 이전에 기록된 고객 정보만 후보 특징으로 검토한다.
- **사용하면 안 되는 정보:** `order_status`, `is_cancelled`, 취소 사유·취소 시각처럼 예측 시점 이후에 알 수 있는 정보는 입력 특징에서 제외한다. `order_id`, `customer_id`, `product_id`도 식별자이므로 제외한다.

주문 상태를 입력 특징으로 사용하면 정답을 미리 알려주는 데이터 누수가 발생한다. 따라서 컬럼이 데이터에 존재하는지뿐 아니라, 실제 예측 시점에 알 수 있는 정보인지 확인해야 한다.

In [2]:
# 이번 분류에 사용할 두 가지 주문 상태만 남긴다.
target_orders = orders.loc[
    orders["order_status"].isin(["completed", "cancelled"])
].copy()

# 취소 주문은 1, 완료 주문은 0으로 표시한다.
target_orders["is_cancelled"] = (
    target_orders["order_status"] == "cancelled"
).astype(int)

# 제외 건수와 타깃 분포를 확인한다.
print("모델링 대상:", len(target_orders))
print("제외한 refunded 주문:", len(orders) - len(target_orders))
print("\n타깃별 건수:")
print(target_orders["is_cancelled"].value_counts().sort_index())
print("\n타깃별 비율(%):")
print((target_orders["is_cancelled"].value_counts(normalize=True).sort_index() * 100).round(2))

모델링 대상: 243
제외한 refunded 주문: 57

타깃별 건수:
is_cancelled
0    180
1     63
Name: count, dtype: int64

타깃별 비율(%):
is_cancelled
0    74.07
1    25.93
Name: proportion, dtype: float64


### 결과 관찰

전체 주문 300건 중 `completed`는 180건, `cancelled`는 63건, `refunded`는 57건이다. 이번 분석에서는 `refunded` 57건을 제외해 총 243건을 모델링 대상으로 정했다. 타깃 `is_cancelled=0`은 180건(74.07%), `is_cancelled=1`은 63건(25.93%)이다.

### 나의 해석과 판단

완료 주문이 취소 주문보다 많다. 모델이 모든 주문을 완료로만 예측해도 전체 데이터 기준 약 74%를 맞힐 수 있으므로, 이후 모델을 평가할 때 accuracy만으로 취소 탐지 성능을 판단하지 않겠다. 취소 주문의 Precision, Recall, F1을 함께 확인하겠다.

### 업무·분석적 의미

주문 생성 직후 취소 가능성이 있는 주문을 찾아내는 것이 목적이다. 다만 취소 주문을 놓치는 경우(FN)와 정상 주문을 취소 위험으로 잘못 표시하는 경우(FP)의 비용은 이후 평가 결과와 업무 조건을 함께 보고 판단해야 한다.

### 한계와 추가 확인 사항

현재 확인한 것은 주문 상태 분포이며, 아직 예측 성능을 측정한 것은 아니다. `refunded` 주문은 분석 대상에서 제외했으므로 이 모델의 결과를 환불 주문에 적용할 수 없다. 후보 특징이 주문 생성 직후 실제로 사용 가능한지도 Feature Audit에서 확인해야 한다.

### STEP 2. 주문 단위 특징과 계산 관계 검증

In [3]:
# STEP 2-1. 전처리된 주문상품 파일을 읽는다.
order_items = pd.read_csv(
    project_root / "data" / "processed" / "order_items_clean.csv"
)

# 계산과 집계에 사용할 컬럼이 있는지 확인한다.
print("주문상품 데이터 크기:", order_items.shape)
print("컬럼:", order_items.columns.tolist())
display(order_items.head())

주문상품 데이터 크기: (752, 6)
컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,39,2.0,172000.0,344000.0
1,2,2,26,4.0,79000.0,316000.0
2,3,2,46,3.0,87000.0,261000.0
3,4,2,80,1.0,187000.0,187000.0
4,5,2,91,5.0,125000.0,625000.0


In [4]:
import numpy as np

# 수량과 단가가 비어 있거나 0 이하인 행을 센다.
invalid_quantity = (
    order_items["quantity"].isna() | (order_items["quantity"] <= 0)
).sum()
invalid_price = (
    order_items["unit_price"].isna() | (order_items["unit_price"] <= 0)
).sum()

# 각 상품 행의 금액이 수량 × 단가와 일치하는지 확인한다.
expected_total = order_items["quantity"] * order_items["unit_price"]
line_total_mismatch = (~np.isclose(
    order_items["line_total"],
    expected_total,
    rtol=0,
    atol=0.01,
    equal_nan=False,
)).sum()

print("수량 오류 행:", invalid_quantity)
print("단가 오류 행:", invalid_price)
print("line_total 계산 불일치 행:", line_total_mismatch)

# 계산 관계가 정상일 때만 주문별 특징을 만든다.
assert invalid_quantity == 0
assert invalid_price == 0
assert line_total_mismatch == 0

order_features = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),       # 주문상품 행 수
        total_quantity=("quantity", "sum"),        # 주문한 총 수량
        order_amount=("line_total", "sum"),        # 주문 총금액
    )
)

print("\n주문별 특징 행 수:", len(order_features))
print("중복 order_id 수:", order_features["order_id"].duplicated().sum())
display(order_features.head())

수량 오류 행: 0
단가 오류 행: 0
line_total 계산 불일치 행: 0

주문별 특징 행 수: 301
중복 order_id 수: 0


,order_id,item_count,total_quantity,order_amount
0,1,1,2.0,344000.0
1,2,4,13.0,1389000.0
2,3,3,7.0,418000.0
3,4,1,2.0,116000.0
4,5,1,5.0,965000.0


### STEP 2. 주문 단위 특징 생성 결과

#### 결과 관찰

주문상품 데이터는 752행이다. 수량 오류, 단가 오류, `line_total = quantity × unit_price` 계산 불일치는 모두 0건이었다. `order_id`별로 집계한 결과 301행이 생성됐고, 집계 결과의 중복 `order_id`는 0건이었다.

#### 나의 해석과 판단

주문상품 여러 행을 주문 1건당 1행으로 바꾸고 `item_count`, `total_quantity`, `order_amount`를 만들었다. 금액 계산은 확인했지만, 주문 테이블은 300행이고 주문별 특징은 301행이므로 두 테이블의 ID가 정확히 연결되는지는 아직 확인되지 않았다.

#### 업무·분석적 의미

이 특징들은 주문 생성 직후의 상품 수, 수량, 금액을 주문 단위로 비교하는 데 사용할 수 있다. 현재 결과만으로 이 특징들이 취소의 원인이라고 해석하지 않는다.

#### 한계와 추가 확인 사항

다음 단계에서 주문과 주문별 특징의 `order_id`를 비교하고, 매칭되지 않는 행이 있는지 확인해야 한다. 매칭되지 않는 값을 이유 없이 0으로 채우지 않겠다.

### STEP 3. 병합 관계와 Feature Audit 확인

In [6]:
# 양쪽 테이블에서 order_id가 중복되는지 먼저 확인한다.
print("주문 행 수:", len(orders))
print("주문 ID 중복 수:", orders["order_id"].duplicated().sum())
print("주문별 특징 행 수:", len(order_features))
print("주문별 특징 ID 중복 수:", order_features["order_id"].duplicated().sum())

# 주문에는 있지만 상품 특징이 없는 ID를 찾는다.
orders_without_items = orders.loc[
    ~orders["order_id"].isin(order_features["order_id"]),
    ["order_id", "order_status"],
]

# 상품 특징에는 있지만 주문 테이블에 없는 ID를 찾는다.
items_without_orders = order_features.loc[
    ~order_features["order_id"].isin(orders["order_id"]),
    ["order_id", "item_count", "total_quantity", "order_amount"],
]

print("\n상품 특징이 없는 주문:", len(orders_without_items))
print("주문 테이블에 없는 상품 특징:", len(items_without_orders))
display(orders_without_items)
display(items_without_orders)

주문 행 수: 300
주문 ID 중복 수: 0
주문별 특징 행 수: 301
주문별 특징 ID 중복 수: 0

상품 특징이 없는 주문: 1
주문 테이블에 없는 상품 특징: 2


,order_id,order_status
174,175,completed


,order_id,item_count,total_quantity,order_amount
299,9991,1,5.0,890000.0
300,9992,1,1.0,32000.0


In [7]:
import sys

print("노트북 실행 위치:", Path.cwd())
print("실제로 읽은 프로젝트 폴더:", project_root.resolve())
print("실제로 읽은 주문 파일:", (project_root / "data" / "processed" / "orders_clean.csv").resolve())
print("사용 중인 Python:", sys.executable)

노트북 실행 위치: c:\dev\llm-data-analysis-course\notebooks
실제로 읽은 프로젝트 폴더: C:\dev\llm-data-analysis-course
실제로 읽은 주문 파일: C:\dev\llm-data-analysis-course\data\processed\orders_clean.csv
사용 중인 Python: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
